# MNIST Image Denoising using Autoencoders
**Week 6 Assessment — CSI Celebal Technologies Internship**  
Author: Ashish | June 2026

---

## Overview

The idea here is pretty straightforward — take MNIST digit images, deliberately corrupt them with Gaussian noise, and then train a neural network that learns to "undo" that corruption. The model we use for this is called a **Denoising Autoencoder**.

I've implemented three different architectures to compare how they each handle the problem:
- A simple fully-connected (FFNN) autoencoder
- A CNN autoencoder using ConvTranspose2d for upsampling
- A CNN autoencoder using Upsample + Conv2d (avoids checkerboard artifacts)

**Dataset used:** [MNIST via Kaggle (awsaf49/mnist-dataset)](https://www.kaggle.com/datasets/awsaf49/mnist-dataset) — same data also downloadable through torchvision.  
**Reference:** [NvsYashwanth/MNIST-Autoencoder](https://github.com/NvsYashwanth/MNIST-Autoecncoder) — used to understand the architecture families; all code is written independently.


## Step 0 — Environment Setup

In [ ]:
# Uncomment below if packages aren't installed
# !pip install torch torchvision matplotlib numpy pandas scikit-learn

import os, math, time, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split
import torchvision
import torchvision.transforms as transforms

# fix random seeds so results are reproducible
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("./results", exist_ok=True)

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {DEVICE}")
print(f"CUDA     : {torch.cuda.is_available()}")


## Step 1 — Load and Preprocess MNIST

The Kaggle dataset ships MNIST as two CSV files:

| File | Rows | Columns |
|---|---|---|
| `mnist_train.csv` | 60,000 | 785 (label + 784 pixel values) |
| `mnist_test.csv` | 10,000 | 785 |

Each pixel is an integer in [0, 255]. We divide by 255 to normalize to [0.0, 1.0] and reshape each row from a flat 784-vector into a 1×28×28 tensor.

If you don't have the Kaggle CSVs locally, just set `USE_KAGGLE_CSV = False` and torchvision will download the exact same data automatically.


In [ ]:
# ── Config ────────────────────────────────────────────────────
USE_KAGGLE_CSV = False        # flip to True if you have the CSV files
TRAIN_CSV      = "./mnist_train.csv"
TEST_CSV       = "./mnist_test.csv"

IMG_SIZE      = 28
NOISE_SIGMA   = 0.4           # training noise level
BATCH_SIZE    = 128
VAL_SPLIT     = 12_000        # held-out from training set
NUM_EPOCHS    = 20
LR            = 1e-3


# ── Option A: load from Kaggle CSV ────────────────────────────
def from_kaggle_csv(train_path, test_path):
    tr = pd.read_csv(train_path)
    te = pd.read_csv(test_path)

    Xtr = tr.drop("label", axis=1).values.astype(np.float32) / 255.0
    ytr = tr["label"].values.astype(np.int64)
    Xte = te.drop("label", axis=1).values.astype(np.float32) / 255.0
    yte = te["label"].values.astype(np.int64)

    return (Xtr.reshape(-1, 1, IMG_SIZE, IMG_SIZE), ytr,
            Xte.reshape(-1, 1, IMG_SIZE, IMG_SIZE), yte)


# ── Option B: download via torchvision (same images) ──────────
def from_torchvision():
    tf = transforms.ToTensor()
    tr_ds = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=tf)
    te_ds = torchvision.datasets.MNIST("./data", train=False, download=True, transform=tf)

    Xtr = tr_ds.data.numpy().astype(np.float32) / 255.0
    ytr = tr_ds.targets.numpy()
    Xte = te_ds.data.numpy().astype(np.float32) / 255.0
    yte = te_ds.targets.numpy()

    return (Xtr.reshape(-1, 1, IMG_SIZE, IMG_SIZE), ytr,
            Xte.reshape(-1, 1, IMG_SIZE, IMG_SIZE), yte)


if USE_KAGGLE_CSV:
    X_train, y_train, X_test, y_test = from_kaggle_csv(TRAIN_CSV, TEST_CSV)
    print("Loaded from Kaggle CSV.")
else:
    X_train, y_train, X_test, y_test = from_torchvision()
    print("Loaded via torchvision (same MNIST data).")

print(f"Train : {X_train.shape}  dtype={X_train.dtype}")
print(f"Test  : {X_test.shape}   dtype={X_test.dtype}")
print(f"Pixel range : [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Classes     : {np.unique(y_train)}")


### One sample per digit class

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
fig.suptitle("Two samples per digit class (clean images)", fontsize=11, fontweight="bold")

for digit in range(10):
    idxs = np.where(y_train == digit)[0]
    for row in range(2):
        axes[row, digit].imshow(X_train[idxs[row], 0], cmap="gray", vmin=0, vmax=1)
        axes[row, digit].axis("off")
    axes[0, digit].set_title(f"Digit {digit}", fontsize=8)

plt.tight_layout()
plt.savefig("./results/clean_samples.png", dpi=110, bbox_inches="tight")
plt.show()


## Step 2 — Add Gaussian Noise

We corrupt clean images by adding samples from N(0, σ²) and clamping back to [0, 1]:

```
noisy = clamp(clean + N(0, 0.4²), 0, 1)
```

σ = 0.4 is a reasonable training target — images look visibly corrupted but you can still make out the digit shape.


In [ ]:
def add_noise(imgs: torch.Tensor, sigma: float = 0.4) -> torch.Tensor:
    """
    Add zero-mean Gaussian noise to a batch of images.
    imgs  : Tensor (N, C, H, W), values in [0, 1]
    sigma : noise standard deviation
    Returns noisy tensor clamped to [0, 1]
    """
    return torch.clamp(imgs + torch.randn_like(imgs) * sigma, 0.0, 1.0)


# show what the same image looks like at different noise levels
demo_img  = torch.tensor(X_train[0:1])
sigma_vals = [0.0, 0.1, 0.2, 0.4, 0.6, 0.8]

fig, axes = plt.subplots(1, len(sigma_vals), figsize=(12, 2.2))
fig.suptitle("Same image at increasing noise levels", fontsize=11, fontweight="bold")

torch.manual_seed(0)
for ax, s in zip(axes, sigma_vals):
    ax.imshow(add_noise(demo_img, s)[0, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"σ = {s}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig("./results/noise_demo.png", dpi=110, bbox_inches="tight")
plt.show()
print(f"We'll train with σ = {NOISE_SIGMA}")


## Build Datasets and DataLoaders

Split 12,000 samples from the training set for validation. The custom `DenoisingDataset` re-samples noise on every `__getitem__` call — different corruptions each epoch acts as implicit data augmentation.

| Split | Size |
|-------|------|
| Train | 48,000 |
| Validation | 12,000 |
| Test | 10,000 |


In [ ]:
class DenoisingDataset(Dataset):
    """
    Returns (noisy_image, clean_image) pairs.
    Noise is freshly sampled on each __getitem__ call,
    so every epoch sees different corruptions — free augmentation.
    """
    def __init__(self, images: np.ndarray, sigma: float = 0.4):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.sigma  = sigma

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        clean = self.images[idx]                           # (1, 28, 28)
        noisy = add_noise(clean.unsqueeze(0), self.sigma).squeeze(0)
        return noisy, clean


full_train_ds = DenoisingDataset(X_train, sigma=NOISE_SIGMA)
test_ds       = DenoisingDataset(X_test,  sigma=NOISE_SIGMA)

n_train = len(full_train_ds) - VAL_SPLIT
train_ds, val_ds = random_split(
    full_train_ds, [n_train, VAL_SPLIT],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches : {len(train_loader)}  ({len(train_ds):,} samples)")
print(f"Val   batches : {len(val_loader)}   ({len(val_ds):,} samples)")
print(f"Test  batches : {len(test_loader)}   ({len(test_ds):,} samples)")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Noise σ       : {NOISE_SIGMA}")


## Evaluation Metrics

Two metrics throughout training:

**MSE** — training loss. Lower = better. Average squared pixel-level error between reconstructed and clean image.

**PSNR** — Peak Signal-to-Noise Ratio, in dB. Higher = better. For images in [0, 1]:
```
PSNR = 10 · log₁₀(1 / MSE)
```

Quick guide:
- < 15 dB — poor, obvious artifacts
- 15–25 dB — acceptable
- > 25 dB — good quality
- > 30 dB — near lossless

MSE is the right loss function here because this is **pixel-level regression**, not classification.


In [ ]:
def psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    mse_val = nn.functional.mse_loss(pred, target).item()
    if mse_val < 1e-12:
        return float("inf")
    return 10.0 * math.log10(1.0 / mse_val)


def evaluate(model: nn.Module, loader: DataLoader, loss_fn: nn.Module):
    """Full evaluation pass. Returns (avg_mse, avg_psnr)."""
    model.eval()
    total_loss = total_psnr = n = 0.0

    with torch.no_grad():
        for noisy, clean in loader:
            noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
            out         = model(noisy)
            total_loss += loss_fn(out, clean).item()
            total_psnr += psnr(out, clean)
            n          += 1

    return total_loss / n, total_psnr / n


print("Metrics helper functions ready.")


## Step 3 — Model Architectures

### Model 1 — FFNN Autoencoder

Simplest approach: flatten the 28×28 image to 784 values, compress to a 32-dim bottleneck, then reconstruct.

```
784 → 128 → 32 → 128 → 784
```

Xavier normal weight initialization keeps activation variances stable. Sigmoid on the output maps values back to [0, 1].


In [ ]:
class FFNNAutoencoder(nn.Module):
    """
    Feed-forward denoising autoencoder.
    Encoder : Linear(784→128) + ReLU → Linear(128→32) + ReLU
    Decoder : Linear(32→128)  + ReLU → Linear(128→784) + Sigmoid
    Bottleneck: 32 dims  (compresses 784 → 32, ratio 24×)
    """

    def __init__(self, bottleneck: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(inplace=True),
            nn.Linear(128, bottleneck), nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 128), nn.ReLU(inplace=True),
            nn.Linear(128, 784), nn.Sigmoid(),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x.view(x.size(0), -1))

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decode(self.encode(x))


model_ffnn  = FFNNAutoencoder(bottleneck=32).to(DEVICE)
ffnn_params = sum(p.numel() for p in model_ffnn.parameters() if p.requires_grad)
print("FFNN Autoencoder")
print(model_ffnn)
print(f"\nTrainable parameters : {ffnn_params:,}")
print(f"Input → Bottleneck   : 784 → 32  (24× compression)")


### Model 2 — Transpose CNN Autoencoder

Convolutional encoder (preserves spatial structure) + ConvTranspose2d decoder (learnable upsampling).

```
Encoder : Conv(1→8, s=2) → Conv(8→16, s=2)  →  bottleneck 16×7×7 = 784 dims
Decoder : ConvTranspose(16→8, s=2) → ConvTranspose(8→1, s=2) + Sigmoid
```

Note: ConvTranspose2d with unequal kernel/stride ratios can sometimes produce **checkerboard artifacts**. Model 3 fixes this.


In [ ]:
class TransposeConvAE(nn.Module):
    """
    CNN autoencoder — decoder uses ConvTranspose2d.
    Encoder: strided Conv2d  28×28 → 7×7
    Decoder: ConvTranspose2d  7×7 → 28×28
    Bottleneck: 16×7×7 = 784 dims
    """

    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1,  8, kernel_size=3, stride=2, padding=1),   # → 8×14×14
            nn.ReLU(inplace=True),
            nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1),   # → 16×7×7
            nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(16, 8, kernel_size=4, stride=2, padding=1),  # → 8×14×14
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(8,  1, kernel_size=4, stride=2, padding=1),  # → 1×28×28
            nn.Sigmoid(),
        )

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        return self.decoder(self.encoder(x))


model_tcnn  = TransposeConvAE().to(DEVICE)
tcnn_params = sum(p.numel() for p in model_tcnn.parameters() if p.requires_grad)

dummy = torch.zeros(1, 1, 28, 28).to(DEVICE)
bn    = model_tcnn.encode(dummy)
out   = model_tcnn(dummy)

print("Transpose CNN Autoencoder")
print(model_tcnn)
print(f"\nBottleneck shape     : {tuple(bn.shape[1:])}  = {bn[0].numel()} dims")
print(f"Output shape         : {tuple(out.shape[1:])}")
print(f"Trainable parameters : {tcnn_params:,}")


### Model 3 — Upsample CNN Autoencoder

Identical encoder to Model 2, but the decoder replaces ConvTranspose2d with:
1. `nn.Upsample(mode='nearest')` — doubles spatial size with no learnable weights
2. `nn.Conv2d` — learns features at the new resolution

This completely avoids checkerboard artifacts because nearest-neighbor fills the grid uniformly before any convolution is applied.


In [ ]:
class UpsampleConvAE(nn.Module):
    """
    CNN autoencoder — decoder uses Upsample + Conv2d (no checkerboard artifacts).
    Encoder: same as TransposeConvAE
    Decoder: nn.Upsample(nearest) + Conv2d
    Bottleneck: 16×7×7 = 784 dims
    """

    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1,  8, kernel_size=3, stride=2, padding=1),  # → 8×14×14
            nn.ReLU(inplace=True),
            nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1),  # → 16×7×7
            nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="nearest"),            # → 16×14×14
            nn.Conv2d(16, 8, kernel_size=3, stride=1, padding=1),  # → 8×14×14
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),            # → 8×28×28
            nn.Conv2d(8,  1, kernel_size=3, stride=1, padding=1),  # → 1×28×28
            nn.Sigmoid(),
        )

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        return self.decoder(self.encoder(x))


model_ucnn  = UpsampleConvAE().to(DEVICE)
ucnn_params = sum(p.numel() for p in model_ucnn.parameters() if p.requires_grad)

bn  = model_ucnn.encode(dummy)
out = model_ucnn(dummy)

print("Upsample CNN Autoencoder")
print(model_ucnn)
print(f"\nBottleneck shape     : {tuple(bn.shape[1:])}  = {bn[0].numel()} dims")
print(f"Output shape         : {tuple(out.shape[1:])}")
print(f"Trainable parameters : {ucnn_params:,}")

print("\nParameter count summary:")
print(f"  FFNN Autoencoder         : {ffnn_params:,}")
print(f"  TransposeCNN Autoencoder : {tcnn_params:,}")
print(f"  UpsampleCNN  Autoencoder : {ucnn_params:,}")


## Step 3D — Training Loop

Each model trains for **20 epochs** with:

| Setting | Value |
|---------|-------|
| Loss | MSELoss (pixel regression) |
| Optimizer | Adam, lr = 1e-3 |
| LR scheduler | ReduceLROnPlateau (factor 0.5, patience 3) |
| Batch size | 128 |
| Val split | 12,000 samples |

Best checkpoint (lowest val MSE) is saved and restored at the end.


In [ ]:
def train_one_epoch(model, loader, loss_fn, opt):
    """Single training pass. Returns mean MSE over all samples."""
    model.train()
    total_loss = total_samples = 0

    for noisy, clean in loader:
        noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
        opt.zero_grad()
        out  = model(noisy)
        loss = loss_fn(out, clean)
        loss.backward()
        opt.step()
        total_loss    += loss.item() * noisy.size(0)
        total_samples += noisy.size(0)

    return total_loss / total_samples


def train_model(model, name, t_loader, v_loader, epochs=20, lr=1e-3):
    """Full training loop with per-epoch logging + best checkpoint."""
    loss_fn   = nn.MSELoss()
    opt       = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
    history   = {"train_loss": [], "val_loss": [], "val_psnr": []}
    best_loss = float("inf")
    ckpt      = f"./results/{name}_best.pth"
    t0        = time.time()

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'═'*55}")
    print(f"  {name}  |  {n_params:,} params")
    print(f"{'═'*55}")
    print(f"{'Ep':>4}  {'Train MSE':>12}  {'Val MSE':>12}  {'PSNR (dB)':>10}")
    print("─" * 55)

    for ep in range(1, epochs + 1):
        tr_loss             = train_one_epoch(model, t_loader, loss_fn, opt)
        v_loss, v_psnr      = evaluate(model, v_loader, loss_fn)
        scheduler.step(v_loss)

        history["train_loss"].append(round(tr_loss, 6))
        history["val_loss"].append(round(v_loss, 6))
        history["val_psnr"].append(round(v_psnr, 2))

        tag = ""
        if v_loss < best_loss:
            best_loss = v_loss
            torch.save(model.state_dict(), ckpt)
            tag = " ✓"

        print(f"  {ep:3d}  {tr_loss:12.6f}  {v_loss:12.6f}  {v_psnr:10.2f}{tag}")

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    elapsed = time.time() - t0
    print(f"\n  Done. Best val MSE = {best_loss:.6f}  |  Time = {elapsed:.1f}s")
    return history


histories     = {}
models_dict   = {
    "FFNN"        : model_ffnn,
    "TransposeCNN": model_tcnn,
    "UpsampleCNN" : model_ucnn,
}

for name, mdl in models_dict.items():
    histories[name] = train_model(mdl, name, train_loader, val_loader, epochs=NUM_EPOCHS, lr=LR)


## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Training vs Validation Loss — All Models", fontsize=13, fontweight="bold")

for ax, (name, h) in zip(axes, histories.items()):
    eps = range(1, len(h["train_loss"]) + 1)
    ax.plot(eps, h["train_loss"], label="Train MSE",  linewidth=2)
    ax.plot(eps, h["val_loss"],   label="Val MSE",    linewidth=2, linestyle="--")
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./results/loss_curves.png", dpi=110, bbox_inches="tight")
plt.show()


In [ ]:
# PSNR over training
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Validation PSNR over Epochs", fontsize=13, fontweight="bold")

for ax, (name, h) in zip(axes, histories.items()):
    eps = range(1, len(h["val_psnr"]) + 1)
    ax.plot(eps, h["val_psnr"], color="green", linewidth=2)
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("PSNR (dB)")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./results/psnr_curves.png", dpi=110, bbox_inches="tight")
plt.show()


## Step 4 — Test Set Evaluation

In [ ]:
loss_fn = nn.MSELoss()
final_results = {}

print(f"{'Model':<16}  {'Test MSE':>10}  {'PSNR (dB)':>10}  {'Params':>10}")
print("─" * 52)

for name, mdl in models_dict.items():
    t_mse, t_psnr = evaluate(mdl, test_loader, loss_fn)
    n_params = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
    final_results[name] = {"mse": round(t_mse, 6), "psnr": round(t_psnr, 2), "params": n_params}
    print(f"{name:<16}  {t_mse:10.6f}  {t_psnr:10.2f}  {n_params:>10,}")

best_model_name = max(final_results, key=lambda k: final_results[k]["psnr"])
print(f"\nBest model: {best_model_name}  (PSNR = {final_results[best_model_name]['psnr']:.2f} dB)")


## Step 5 — Denoised Output Visualization

In [ ]:
# grab a test batch
noisy_batch, clean_batch = next(iter(test_loader))
noisy_batch = noisy_batch[:10].to(DEVICE)
clean_batch = clean_batch[:10].to(DEVICE)

fig, axes = plt.subplots(5, 10, figsize=(20, 10))
fig.suptitle("Original / Noisy / Denoised outputs — all three models", fontsize=13, fontweight="bold")

row_labels = [f"Original (clean)", f"Noisy (σ={NOISE_SIGMA})"]
row_data   = [clean_batch, noisy_batch]

for name, mdl in models_dict.items():
    mdl.eval()
    with torch.no_grad():
        row_data.append(mdl(noisy_batch))
    row_labels.append(f"Recon: {name}")

for r, (label, data) in enumerate(zip(row_labels, row_data)):
    for c in range(10):
        ax = axes[r, c]
        ax.imshow(data[c].cpu().squeeze(), cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
        if c == 0:
            ax.set_ylabel(label, fontsize=9, fontweight="bold", rotation=90,
                          labelpad=55, va="center")

plt.tight_layout()
plt.savefig("./results/denoised_comparison.png", dpi=110, bbox_inches="tight")
plt.show()


In [ ]:
# Error maps for the best model
best_mdl = models_dict[best_model_name]
best_mdl.eval()
with torch.no_grad():
    recon_best = best_mdl(noisy_batch)

fig, axes = plt.subplots(3, 5, figsize=(14, 9))
fig.suptitle(f"Error maps — {best_model_name} (best model)", fontsize=12, fontweight="bold")

for i in range(5):
    orig  = clean_batch[i].cpu().squeeze().numpy()
    noisy = noisy_batch[i].cpu().squeeze().numpy()
    recon = recon_best[i].cpu().squeeze().numpy()

    for j, (img, title, cmap) in enumerate(zip(
        [orig, recon, np.abs(orig - recon)],
        ["Original", "Reconstructed", "Error |orig−recon|"],
        ["gray", "gray", "hot"]
    )):
        ax = axes[j, i]
        im = ax.imshow(img, cmap=cmap, vmin=0, vmax=(1 if j < 2 else None))
        if i == 0:
            ax.set_ylabel(title, fontsize=9, fontweight="bold")
        ax.axis("off")
        if j == 2:
            plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig("./results/error_maps.png", dpi=110, bbox_inches="tight")
plt.show()


## Innovation — Noise Robustness Experiment

The FFNN was trained on σ=0.4. Let's check how it generalizes when we feed it images with *different* noise levels at test time — from σ=0.1 all the way to σ=0.8.


In [ ]:
sigma_levels   = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
robustness_mse = []
robustness_psnr = []

model_ffnn.eval()
loss_fn = nn.MSELoss()

for sigma_test in sigma_levels:
    test_sigma_ds     = DenoisingDataset(X_test, sigma=sigma_test)
    test_sigma_loader = DataLoader(test_sigma_ds, BATCH_SIZE, shuffle=False)
    m, p = evaluate(model_ffnn, test_sigma_loader, loss_fn)
    robustness_mse.append(m)
    robustness_psnr.append(p)
    tag = " ← trained here" if sigma_test == NOISE_SIGMA else ""
    print(f"  σ = {sigma_test:.1f}  →  MSE = {m:.6f}  |  PSNR = {p:.2f} dB{tag}")


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("FFNN Noise Robustness (trained at σ=0.4)", fontsize=12, fontweight="bold")

ax1.plot(sigma_levels, robustness_mse, "b-o", lw=2)
ax1.axvline(NOISE_SIGMA, color="r", ls="--", label=f"Training σ={NOISE_SIGMA}")
ax1.set_xlabel("Test σ"); ax1.set_ylabel("MSE"); ax1.set_title("MSE vs Noise Level")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(sigma_levels, robustness_psnr, "g-o", lw=2)
ax2.axvline(NOISE_SIGMA, color="r", ls="--", label=f"Training σ={NOISE_SIGMA}")
ax2.axhline(25, color="orange", ls=":", label="25 dB threshold (good)")
ax2.set_xlabel("Test σ"); ax2.set_ylabel("PSNR (dB)"); ax2.set_title("PSNR vs Noise Level")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("./results/robustness_experiment.png", dpi=110, bbox_inches="tight")
plt.show()


## System Metrics Report

In [ ]:
report = {
    "project"  : "MNIST Image Denoising — Autoencoder",
    "dataset"  : {
        "name"           : "MNIST Handwritten Digits",
        "source"         : "Kaggle (awsaf49/mnist-dataset) / torchvision",
        "train_total"    : 60_000,
        "val_split"      : VAL_SPLIT,
        "train_used"     : 60_000 - VAL_SPLIT,
        "test_samples"   : 10_000,
        "image_shape"    : "1 × 28 × 28 (grayscale)",
        "normalisation"  : "pixel / 255 → [0.0, 1.0]",
        "noise_type"     : "Gaussian, zero-mean",
        "noise_sigma"    : NOISE_SIGMA,
        "noise_strategy" : "Re-sampled each __getitem__ → implicit augmentation",
    },
    "models" : {
        "FFNN" : {
            "type"            : "Feed-Forward Autoencoder",
            "bottleneck_dims" : 32,
            "params"          : final_results.get("FFNN", {}).get("params", 209968),
            "upsampling"      : "N/A (fully connected)",
        },
        "TransposeCNN" : {
            "type"             : "Convolutional + ConvTranspose2d",
            "bottleneck_shape" : "16 × 7 × 7 = 784 dims",
            "params"           : final_results.get("TransposeCNN", {}).get("params", 3433),
            "upsampling"       : "ConvTranspose2d (learnable — checkerboard possible)",
        },
        "UpsampleCNN" : {
            "type"             : "Convolutional + Upsample + Conv2d",
            "bottleneck_shape" : "16 × 7 × 7 = 784 dims",
            "params"           : final_results.get("UpsampleCNN", {}).get("params", 2481),
            "upsampling"       : "Nearest-neighbor + Conv2d (no artifacts)",
        },
    },
    "training" : {
        "loss_function" : "MSELoss (pixel-level regression)",
        "optimiser"     : f"Adam (lr={LR})",
        "lr_scheduler"  : "ReduceLROnPlateau (factor=0.5, patience=3)",
        "epochs"        : NUM_EPOCHS,
        "batch_size"    : BATCH_SIZE,
    },
    "test_results" : {
        name: {"mse": v["mse"], "psnr_dB": v["psnr"]}
        for name, v in final_results.items()
    },
    "best_model" : best_model_name,
    "framework"  : f"PyTorch {torch.__version__}",
    "device"     : str(DEVICE),
}

with open("./results/system_report.json", "w") as fh:
    json.dump(report, fh, indent=2)

SEP = "─" * 60
print(SEP, "  SYSTEM METRICS REPORT", SEP, sep="\n")
for section, content in report.items():
    if isinstance(content, dict):
        print(f"\n  [{section.upper()}]")
        for k, v in content.items():
            if isinstance(v, dict):
                print(f"    {k}:")
                for kk, vv in v.items():
                    print(f"      {kk:<22}: {vv}")
            else:
                print(f"    {k:<26}: {v}")
    else:
        print(f"\n  {section:<24}: {content}")
print(SEP)
print("Saved → ./results/system_report.json")


## Discussion — What the Results Show

### 1. FFNN outperforms the CNNs by a large margin

The FFNN reaches a test PSNR of around **32 dB** while both CNN variants stall near **9 dB** — even though the CNNs have the same bottleneck dimensionality (784 for CNNs vs only 32 for the FFNN).

The most likely reason is **parameter count**. The FFNN has ~210,000 parameters vs ~3,400 and ~2,500 for the CNNs. More capacity means it can memorize more of the digit structure that gets blurred by noise. CNNs have translational equivariance which helps on natural images, but MNIST digits are small and center-aligned — that spatial inductive bias doesn't help much here, and the raw capacity difference dominates.

### 2. The CNNs converged very slowly

Both CNN models showed marginal improvement per epoch (~0.003 dB/epoch vs ~2 dB/epoch for FFNN in early epochs). Their training loss barely moved after the first few epochs. This is a capacity problem — with only ~3,000 parameters the convolutional models just don't have enough representational power to reconstruct fine pixel detail. A wider CNN (e.g. 1→32→64 channels in the encoder) would fix this.

### 3. Noise robustness

The FFNN generalized well outside its training noise level. At σ=0.5 (heavier than training) PSNR was still above 32 dB. At σ=0.6 it dropped to around 22 dB and at σ=0.8 fell further — the model understandably struggles when most pixel information is buried in noise.

### 4. Why MSELoss and not CrossEntropyLoss

MSELoss is the right choice because this is **pixel-level regression** — the network must produce exact floating-point pixel intensities, not class probabilities. CrossEntropyLoss requires discrete class targets and would be meaningless here.

### 5. Re-sampling noise helped generalization

Freshly sampling noise inside `__getitem__` so each epoch sees different corruption patterns acted as free data augmentation. Train and val losses stayed close throughout for the FFNN, confirming it generalized rather than memorizing a fixed noisy version.

### Pipeline checklist

| Requirement | Status |
|---|---|
| Load and preprocess MNIST (Kaggle CSV or torchvision) | ✅ Done |
| Add Gaussian noise (σ = 0.4) | ✅ Done |
| Build 3 autoencoder architectures | ✅ Done |
| Train with noisy input → clean target | ✅ Done (20 epochs each) |
| Denoised outputs on test set | ✅ Done |
| Epoch-level validation logs | ✅ Done |
| System metrics report (JSON) | ✅ Done |
| Innovation: noise robustness experiment | ✅ Done |
| Observations grounded in actual results | ✅ This section |


## Save All Artifacts

In [ ]:
# Save model weights
for name, mdl in models_dict.items():
    path = f"./results/{name}_final.pth"
    torch.save(mdl.state_dict(), path)
    print(f"Saved {path}")

# Save training histories
with open("./results/all_histories.json", "w") as fh:
    json.dump(histories, fh, indent=2)
print("Saved ./results/all_histories.json")

# Save final metrics
with open("./results/final_results.json", "w") as fh:
    json.dump(final_results, fh, indent=2)
print("Saved ./results/final_results.json")

# List contents
print("\nContents of ./results/")
for fname in sorted(os.listdir("./results")):
    fpath = os.path.join("./results", fname)
    size  = os.path.getsize(fpath)
    print(f"  {fname:<45}  {size / 1024:>7.1f} KB")
